# Ontology Schema Visualization (Base vs. SULO)

Renders the domain-class diagrams for both KG schemas from `turtles/base_kg.ttl` and
`turtles/sulo_kg.ttl` -- **not** the full imported upstream SULO ontology
(`turtles/sulo.ttl`), so only the classes and relations this project actually defines show
up, no generic upper-ontology overhead.

Schema-level only (classes + object-property relations) -- not the populated instance graph, which
has millions of nodes and wouldn't render meaningfully.

In [16]:
import re
from rdflib import Graph, Namespace, URIRef, RDF, RDFS, OWL
from rdflib.collection import Collection
from graphviz import Digraph

RDFS_RAW = Namespace(str(RDFS))  # base_kg.ttl types properties as rdfs:Property (nonstandard --
                                 # the real term is rdf:Property), so RDFS.Property would raise;
                                 # a plain Namespace skips rdflib's strict DefinedNamespace check.

volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
BASE_TTL = f"{volume}base_kg.ttl"
SULO_TTL = f"{volume}sulo_kg.ttl"

## Parser

Handles both files' conventions in one pass:
- `base_kg.ttl`: plain `rdfs:Class`/`rdfs:Property`, domain/range point directly at local classes.
- `sulo_kg.ttl`: `owl:Class`/`owl:ObjectProperty`, domain/range point at opaque per-property UUID
  classes that resolve to the real local class via `owl:equivalentClass` -> `owl:intersectionOf`.

A relation edge is kept only if BOTH endpoints resolve to a class actually declared in this file's
own local namespace (the file's own `:` prefix, read dynamically -- it's a sub-path of the
upstream `https://w3id.org/sulo/` namespace, not a disjoint one, so simple prefix-exclusion of
`sulo:` doesn't work here). This is what excludes upstream `sulo:` classes and literal-valued
(`xsd:string`/`xsd:dateTime`/`xsd:integer`) datatype properties automatically, without needing to
hardcode which properties are "object" vs "datatype". The class set also excludes blank nodes
(`sulo_kg.ttl`'s anonymous `owl:intersectionOf` wrapper nodes are themselves typed `owl:Class`,
so without this filter they'd show up in the diagram as disconnected garbled-id nodes).

**Every SULO `hasXxx` property is a schema-only shortcut, never a real predicate.** In
`sulo_kg.ttl`, named object properties like `hasAccount`, `hasVehicle`, `hasServiceType`,
`hasSupportLevel`, `hasTask`, ... are each declared via an `owl:equivalentClass` axiom that
expands to a small restriction chain over SULO's actual, reusable core vocabulary --
`hasFeature`/`hasPart`/`isReferredToIn`/`atTime`/`hasParticipant`/`isFeatureOf` -- e.g.
`hasServiceType` unfolds to `Case [hasFeature -> ServiceType]`, and `hasAccount` unfolds to the
two-hop `Case [hasParticipant -> AccountRole [isFeatureOf -> Account]]` (`AccountRole` is a real
class, `rdfs:subClassOf sulo:Role`). `Graph_population.ipynb`'s `DIRECT_OBJECT`/`ROLE_CHAIN`/
`SOLID_LITERAL` patterns confirm this is exactly the real-world split: every instance triple it
ever asserts uses the core predicate, never the `hasXxx` name -- so a diagram edge labeled
`hasServiceType` or `hasCaseType` would show a relation that's declared in the schema but is
never actually asserted in the populated graph. `walk` below always resolves the real
restriction-chain predicate when the schema provides one (which is every SULO object property
except `hasPriorityLevel`, whose domain/range are plain classes with no restriction to recover a
predicate from -- that one edge falls back to its declared name for lack of any better signal).
This also fixes the original bug this notebook was tracking down: naively picking only the first
plain-class member of each `intersectionOf` list collapses `hasAccount`'s chain straight from
`Case` to `Account`, silently skipping `AccountRole` -- it still renders as a node (it's a locally
declared class) but with no edges touching it, so graphviz drifts it off to the side.

A full property-by-property audit against `Graph_population.ipynb`'s actual `sulo_predicate`
values (one audit pass per SULO object/datatype property) found exactly one schema/population
disagreement, since fixed at the source: `hasCountry` was declared in `sulo_kg.ttl` as
`Account [isIn -> Country]`, while the population code always asserted `hasFeature` for that
column. Both the forward and inverse-mirror restriction blocks for `hasCountry` now say
`hasFeature`, matching every other `Account`/`Case`-level `hasFeature` edge and what's actually
in the populated graph. Every other property already matched.

In [ ]:
def local_name(uri):
    return re.split(r'[/#]', str(uri).rstrip('/'))[-1]


def load_schema(path):
    g = Graph()
    g.parse(path, format='turtle')

    # The file's own local ":" prefix (e.g. .../ontology-schema/<uuid>/) is itself a sub-path
    # of the upstream https://w3id.org/sulo/ namespace, so excluding "starts with SULO" would
    # wrongly exclude every local class too. Match the local namespace positively instead.
    local_ns = str(dict(g.namespaces())[''])

    classes = {
        s for s, _, o in g.triples((None, RDF.type, None))
        if o in (OWL.Class, RDFS.Class)
        and isinstance(s, URIRef)  # excludes anonymous owl:intersectionOf wrapper nodes
        and str(s).startswith(local_ns)
    }

    def intersection_members(node):
        """`node`'s owl:intersectionOf list, whether declared directly on `node` (the shape used
        by nested someValuesFrom targets) or indirectly via owl:equivalentClass (the shape used
        by the top-level domain/range UUID nodes). None if neither is present."""
        inter = g.value(node, OWL.intersectionOf)
        if inter is None:
            eq = g.value(node, OWL.equivalentClass)
            if eq is not None:
                inter = g.value(eq, OWL.intersectionOf)
        return list(Collection(g, inter)) if inter is not None else None

    def walk(node):
        """Resolves a domain/range class-expression node to [(anchor_class, hops), ...].

        `anchor_class` is the class this side plainly denotes. `hops` is the downstream
        owl:Restriction chain from that anchor as [(predicate_local_name, class_uri), ...] --
        empty unless the expression nests a restriction whose someValuesFrom needs its own
        intersectionOf unwrap (SULO's AccountRole/VehicleRole Role pattern, its
        TimeInstant/DurationSec/Amount value-holder pattern, and every plain single-hop
        relation like hasServiceType/hasSupportLevel). A restriction on an inverseOf property
        is a mirror of the forward chain declared on the other side of the same property, so
        it isn't walked. Returns multiple tuples only for owl:unionOf domains (e.g.
        hasPriorityLevel's Case/Task).
        """
        if node is None:
            return []
        if node in classes:
            return [(node, [])]
        union = g.value(node, OWL.unionOf)
        if union is not None:
            out = []
            for member in Collection(g, union):
                out.extend(walk(member))
            return out
        members = intersection_members(node)
        if members is None:
            return []
        anchor = next((m for m in members if m in classes), None)
        if anchor is None:
            return []
        restriction = next((m for m in members if (m, RDF.type, OWL.Restriction) in g), None)
        if restriction is None:
            return [(anchor, [])]
        prop = g.value(restriction, OWL.onProperty)
        if not isinstance(prop, URIRef):  # blank node -> a [owl:inverseOf ...] restriction
            return [(anchor, [])]
        inner = walk(g.value(restriction, OWL.someValuesFrom))
        if not inner:
            return [(anchor, [])]
        inner_anchor, inner_hops = inner[0]
        return [(anchor, [(local_name(prop), inner_anchor)] + inner_hops)]

    edges = set()

    def emit_chain(top, hops):
        chain = [top] + [c for _, c in hops]
        preds = [p for p, _ in hops]
        for c1, p, c2 in zip(chain, preds, chain[1:]):
            edges.add((local_name(c1), p, local_name(c2)))

    prop_types = (OWL.ObjectProperty, OWL.DatatypeProperty, RDF.Property, RDFS_RAW.Property)
    properties = {s for s, _, o in g.triples((None, RDF.type, None)) if o in prop_types}
    for prop in properties:
        prop_name = local_name(prop)
        for dom in g.objects(prop, RDFS.domain):
            for d_top, d_hops in walk(dom) or [(None, [])]:
                rng_results = []
                for rng in g.objects(prop, RDFS.range):
                    rng_results.extend(walk(rng) or [(None, [])])
                for r_top, r_hops in rng_results or [(None, [])]:
                    if d_hops:
                        # The restriction chain is the real predicate SULO actually asserts
                        # (see Graph_population.ipynb's ROLE_CHAIN/SOLID_LITERAL/DIRECT_OBJECT
                        # patterns -- hasFeature/hasPart/isReferredToIn/atTime/hasParticipant/
                        # isFeatureOf, never the "hasXxx" shortcut name) -- prefer it whenever
                        # the schema encodes one, whether that reveals a genuine intermediate
                        # class (AccountRole, 2+ hops) or is a single restriction hop that lands
                        # directly on the plain range (e.g. Case -[hasFeature]-> ServiceType,
                        # not Case -[hasServiceType]-> ServiceType -- "hasServiceType" is never
                        # asserted in the populated graph, only declared as its OWL-equivalent).
                        emit_chain(d_top, d_hops)
                    elif r_hops:
                        emit_chain(r_top, r_hops)
                    elif d_top is not None and r_top is not None:
                        # No restriction info on either side to recover the real predicate from
                        # (e.g. hasPriorityLevel's domain/range are plain classes, no
                        # owl:Restriction at all) -- falling back to the declared property name
                        # is the schema's only available signal here.
                        edges.add((local_name(d_top), prop_name, local_name(r_top)))

    return {local_name(c) for c in classes}, sorted(edges)


base_classes, base_edges = load_schema(BASE_TTL)
sulo_classes, sulo_edges = load_schema(SULO_TTL)

print(f"Base:  {len(base_classes)} classes, {len(base_edges)} relations")
print(f"SULO:  {len(sulo_classes)} classes, {len(sulo_edges)} relations")

## Render

Needs the `dot` binary (the Graphviz Python package is just bindings around it). If the cell below fails with `ExecutableNotFound`, run this once on the driver node, then re-run:
```
%sh apt-get install -y graphviz
```
This is driver-node-scoped and won't persist across cluster restarts unless added to the cluster's `init.sh`.

In [18]:
# %sh which dot || apt-get install -y graphviz

In [19]:
def render_schema(classes, edges, title, out_path):
    dot = Digraph(comment=title, format='png')
    dot.attr(rankdir='LR', fontsize='11', label=title, labelloc='t')
    dot.attr('node', shape='box', style='rounded,filled', fillcolor='#eef3fb', fontsize='11')
    dot.attr('edge', fontsize='9')

    for c in sorted(classes):
        dot.node(c, c)
    for src, prop, dst in edges:
        dot.edge(src, dst, label=prop)

    dot.render(out_path, cleanup=True)
    return dot

render_schema(base_classes, base_edges, "Ad-hoc (Base) Schema", f"{volume}base_schema")
render_schema(sulo_classes, sulo_edges, "SULO Schema", f"{volume}sulo_schema")

## Display

PNGs are saved directly to the volume: `{volume}base_schema.png` / `{volume}sulo_schema.png`.

In [20]:
from IPython.display import Image, display

display(Image(filename=f"{volume}base_schema.png"))
display(Image(filename=f"{volume}sulo_schema.png"))